# Density-sensitive bifiltration on a noisy annulus

This notebook demonstrates how the bifiltered Dowker complex detects topological
features robustly in the presence of outliers.

We sample three point clouds from a thin annulus:

- **X** — clean annulus
- **Y** — annulus with 5% uniform noise inside the hole
- **Z** — uniform disc (no ring structure)

The bifiltration's second axis (witness count / density) suppresses the outlier
points, making the H1 feature of the annulus visible even in the noisy sample Y.

Install: `pip install pyDowker[multipers] scikit-learn`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import multipers as mp
from sklearn.metrics import pairwise_distances
from pyDowker import DowkerComplex

## Sample the point clouds

In [ ]:
def sample_annulus(n, inner_r, outer_r, outlier_frac=0.0, seed=0):
    rng = np.random.default_rng(seed)
    angles = rng.uniform(0, 2 * np.pi, n)
    radii  = np.sqrt(rng.uniform(inner_r**2, outer_r**2, n))
    # replace a fraction of points with uniform noise inside the hole
    mask = rng.random(n) < outlier_frac
    radii[mask] = np.sqrt(rng.uniform(0, inner_r**2, mask.sum()))
    return np.column_stack([radii * np.cos(angles), radii * np.sin(angles)])

n = 100
X = sample_annulus(n, 0.4, 0.5, seed=0)            # clean annulus
Y = sample_annulus(n, 0.4, 0.5, outlier_frac=0.05, seed=1)  # noisy
Z = sample_annulus(n, 0.0, 0.5, seed=2)            # filled disc

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, pts, title in zip(axes, [X, Y, Z],
                           ['X — clean annulus', 'Y — 5% noise', 'Z — filled disc']):
    ax.scatter(pts[:, 0], pts[:, 1], s=8)
    ax.set_title(title)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Compute the bifiltered Dowker complexes

We use each point cloud as both data and witnesses (landmark-free setting).
The relation matrix is the pairwise distance matrix.

In [ ]:
m_max = 10  # maximum witness count

sts = {}
for name, pts in [('X', X), ('Y', Y), ('Z', Z)]:
    D = pairwise_distances(pts)
    sts[name] = DowkerComplex(D).create_multipers_bifiltration(
        max_dimension=2, m_max=m_max
    )
    print(f'{name}: {sts[name].num_simplices} simplices')

## Module approximations

We compute module approximations of the degree-1 persistent homology.
A non-trivial H1 module (a visible 'bar' or 'box') corresponds to the loop
in the annulus.  The second filtration axis (negative witness count) encodes
density — only simplices with many witnesses survive at large $m$.

**Expected outcome:**
- X and Y show a clear H1 feature (the annulus is detected even with noise).
- Z shows no H1 feature (filled disc, no loop).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, st) in zip(axes, sts.items()):
    mma = mp.module_approximation(st)
    mma.plot(degree=1, box=[[0, -m_max], [1.0, 0]], ax=ax)
    ax.set_title(f'{name} — H1 module approximation')
plt.tight_layout()
plt.show()

## One-parameter slices for comparison

Fixing a witness threshold $m = 1$ recovers the standard sublevel-filtered
Dowker complex (equivalent to Čech complex via Dowker duality).

In [ ]:
import gudhi as gd

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, pts) in zip(axes, [('X', X), ('Y', Y), ('Z', Z)]):
    D = pairwise_distances(pts)
    st = DowkerComplex(D).create_simplex_tree(
        filtration='Sublevel', m=1, max_dimension=2
    )
    gd.plot_persistence_diagram(st.persistence(), axes=ax)
    ax.set_title(f'{name} — Sublevel persistence (m=1)')
plt.tight_layout()
plt.show()